# Imports

## Standard Library Imports

In [1]:
from enum import Enum
import io
import re

## Third-party Imports

In [2]:
import fitz
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from matplotlib_venn.layout.venn3 import DefaultLayoutAlgorithm
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns

# Enums

## Value Tracking

In [3]:
class RoomOrder(Enum):
	ABF = 0 # 18a, 18b, 18
	BAF = 1 # 18b, 18a, 18
	AFB = 2 # 18a, 18, 18b
	BFA = 3 # 18b, 18, 18a
	FAB = 4 # 18, 18a, 18b
	FBA = 5 # 18, 18b, 18a
	BA = 6 # 18b, 18a
	BF = 7 # 18b, 18
	FB = 8 # 18, 18b
	FA = 9 # 18, 18a
	AB = 10 # 18a, 18b
	AF = 11 # 18a, 18
	A = 12 # 18a
	B = 13 # 18b
	F = 14 # 18

class GroupComp(Enum):
	INDIVIDUAL = 0
	GROUP = 1

class LecternsViewed(Enum):
	N = "None"
	W = "West"
	M = "Middle"
	E = "East"
	WM = "West, Middle"
	WE = "West, East"
	ME = "Middle, East"
	WME = "West, Middle, East"

class VisitorType(Enum):
	BROWSER = 0
	FOLLOWER = 1
	SEARCHER = 2
	RESEARCHER = 3

class TurnDirection(Enum):
	LEFT = 0
	MIDDLE = 1
	RIGHT = 2
	NONE = 3

class TeamMember(Enum):
	Courtney = 0
	Jerry = 1
	Owen = 2
	Ritvik = 3
	Sofia = 4

class Gender(Enum):
	FEMALE = 0
	MALE = 1

class FirstTurnDirection(Enum):
	LEFT = 0
	MIDDLE = 1
	RIGHT = 2

class DayOfWeek(Enum):
	Sunday = 0
	Monday = 1
	Tuesday = 2
	Wednesday = 3
	Thursday = 4
	Friday = 5
	Saturday = 6

## Label Maps

In [4]:
class BehaviorTitles(Enum):
	chatted_with_visitors = "Chatted with Visitors"
	chatted_with_staff = "Chatted with Staff"
	sat_on_bench = "Sat on a Bench"
	split_from_group = "Split from their Group"
	used_phone = "Used a Cell Phone"
	used_museum_guide = "Used the Museum Guide"
	used_headphones = "Used Headphones"
	read_lectern = "Read Lecterns"
	read_label = "Read Labels"
	took_photos = "Took Photos"
	
class SlipRoomTitles(Enum):
	a18_took_photos = "18A: Took Photos"
	a18_took_videos = "18A: Took Videos"
	a18_viewed_labels = "18A: Viewed Labels"
	b18_touched_casts = "18B: Touched Casts"
	b18_took_photos = "18B: Took Photos"
	b18_viewed_labels = "18B: Viewed Labels"

class Languages(Enum):
	EN = "English"
	ZH_S = "Chinese (Simplified)"
	ZH_T = "Chinese (Traditional)"
	FR = "French"
	DE = "German"
	HI = "Hindi"
	IT = "Italian"
	ES_ES = "Spanish"

# Code

## Export Paths

### Main

In [5]:
# input file paths
visitor_xlsx_path: str = "../../assets/excel_files/observation_tables.xlsx"
survey_xlsx_path: str = "../../assets/excel_files/survey_responses.xlsx"
master_tracker_path: str = "../../assets/tracker_images/Master Visitor Tracker Sheet.pdf"

# sheet names
main_sheet_name: str = "main data"
group_sheet_name_base: str = "object group"
indiv_sheet_name_base: str = "object indiv"

### Bar Charts

In [6]:
bar_behavior_export_path: str = "../../assets/output_files/behavior_bar_chart.png"
bar_subbehavior_export_path: str = "../../assets/output_files/main_room_bar_chart.png"
bar_slip_room_export_path: str = "../../assets/output_files/slip_room_bar_chart.png"

### Box and Whisker Plots

In [7]:
box_observation_export_path: str = "../../assets/output_files/visitor_box_plots.pdf"
box_observation_metadata_export_path: str = "../../assets/metadata_output/visitor_box_plots_metadata.pdf"
box_slip_room_metadata_export_path: str = "../../assets/metadata_output/slip_room_box_plots_metadata.pdf"
box_group_export_path: str = "../../assets/output_files/group_box_plots.pdf"
box_group_metadata_export_path: str = "../../assets/metadata_output/group_box_plots_metadata.pdf"
box_survey_export_path: str = "../../assets/output_files/survey_box_plots.pdf"
box_survey_metadata_export_path: str = "../../assets/metadata_output/survey_box_plots_metadata.pdf"
box_survey_kd_export_path: str = "../../assets/output_files/survey_kd_box_plots.pdf"
box_survey_kd_metadata_export_path: str = "../../assets/metadata_output/survey_kd_box_plots_metadata.pdf"

### Bubble Plot

In [8]:
bubble_export_path: str = "../../assets/output_files/room_18_bubble_plot.png"
bubble_metadata_export_path: str = "../../assets/metadata_output/room_18_bubble_plot_metadata.pdf"

### Pie Charts

In [9]:
pie_observation_export_path: str = "../../assets/output_files/visitor_pie_charts.pdf"
pie_survey_export_path: str = "../../assets/output_files/survey_pie_charts.pdf"

### Statistics

In [10]:
stats_group_export_path: str = "../../assets/output_files/section_stats.pdf"
stats_artifact_export_path: str = "../../assets/output_files/artifact_stats.pdf"

### Venn Diagram

In [11]:
venn_observation_export_path: str = "../../assets/output_files/lectern_venn.pdf"

## Loading and Formatting Dataframes

### Main

In [12]:
visitor_xlsx: pd.ExcelFile = pd.ExcelFile(visitor_xlsx_path)
main_visitor_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=main_sheet_name, index_col=0)

num_visitors: int = main_visitor_df.shape[0]

survey_xlsx: pd.ExcelFile = pd.ExcelFile(survey_xlsx_path)
main_survey_df: pd.DataFrame = pd.read_excel(survey_xlsx, index_col=8)
# isolate row of question names
main_survey_question_names: pd.Series = main_survey_df.iloc[0]
# remove row of question names
main_survey_df = main_survey_df[1:]

C:\Users\ritvi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\styles\stylesheet.py:241: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### Bar Charts

In [13]:
bar_visitor_df = main_visitor_df.assign(
	read_lectern=False,
	read_label=False,
	took_photos=False
)
for i in bar_visitor_df.index.to_list():
	if bar_visitor_df.loc[i, "lecterns_visited"] != "N":
		bar_visitor_df.at[i, "read_lectern"] = True

	v_id: str = bar_visitor_df.loc[i, "visitor_id"]
	groups_sheet_name: str = group_sheet_name_base + " " + "{0:0=3d}".format(v_id)
	group_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=groups_sheet_name, index_col=0)
	for j in group_df.index.to_list():
		if group_df.loc[j, "viewed_labels"] == True:
			bar_visitor_df.at[i, "read_label"] = True
		if group_df.loc[j, "took_photos"] == True:
			bar_visitor_df.at[i, "took_photos"] = True
	
	indiv_sheet_name: str = indiv_sheet_name_base + " " + "{0:0=3d}".format(v_id)
	indiv_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=indiv_sheet_name, index_col=0)
	for j in indiv_df.index.to_list():
		if indiv_df.loc[j, "viewed_labels"] == True:
			bar_visitor_df.at[i, "read_label"] = True
		if indiv_df.loc[j, "took_photos"] == True:
			bar_visitor_df.at[i, "took_photos"] = True

bar_visitor_df.rename(columns={
	"18a_took_photos": "a18_took_photos",
	"18a_took_videos": "a18_took_videos",
	"18a_viewed_labels": "a18_viewed_labels",
	"18b_touched_casts": "b18_touched_casts",
	"18b_took_photos": "b18_took_photos",
	"18b_viewed_labels": "b18_viewed_labels"
}, inplace=True)

### Box Plots

In [14]:
box_visitor_df: pd.DataFrame = main_visitor_df.copy()
def get_full_room_time(series: pd.Series) -> float:
	return (series["total_time"] * 60 - series["18a_total_time"] - series["18b_total_time"]) / 60.0

box_visitor_df["18_total_time"] = main_visitor_df.apply(get_full_room_time, axis=1)

box_groups_df: pd.DataFrame = pd.DataFrame()
for i in box_visitor_df["visitor_id"].values:
	sheet_name: str = group_sheet_name_base + " " + "{0:0=3d}".format(i)
	temp_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=sheet_name, index_col=0)
	col_name: str = "{0:0=3d}".format(i) + "_" + "dwell_time"
	box_groups_df[col_name] = temp_df["dwell_time"]
box_groups_df = box_groups_df.T

box_survey_df: pd.DataFrame = main_survey_df.copy()
def get_knowledge_difference(series: pd.Series) -> int | None:
	if series.isnull()["Q22_1"] == True or series.isnull()["Q23_1"]:
		return None
	return int(series["Q23_1"]) - int(series["Q22_1"])
	
box_survey_df["knowledge_difference"] = box_survey_df.apply(get_knowledge_difference, axis=1)

### Bubble Plot

In [15]:
bubble_groups_dfs: dict[str, pd.DataFrame] = {}
for i in range(0, num_visitors):
	visitor_id: str = "{0:0=3d}".format(main_visitor_df.loc[i, "visitor_id"])
	sheet_name: str = group_sheet_name_base + " " + visitor_id
	group_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=sheet_name, index_col=0)
	bubble_groups_dfs[visitor_id] = group_df

### Pie Charts

In [16]:
pie_visitor_df: pd.DataFrame = main_visitor_df.copy()
pie_survey_df: pd.DataFrame = main_survey_df.copy()
pie_survey_question_names: pd.Series = main_survey_question_names.copy()

for i in pie_survey_question_names.keys():
	pie_survey_question_names[i] = pie_survey_question_names[i].replace(" - Selected Choice", "")
	pie_survey_question_names[i] = pie_survey_question_names[i].replace(", ", ", <br>")
	pie_survey_question_names[i] = pie_survey_question_names[i].replace(": ", ": <br>")

for i in pie_survey_df.index.to_list():
	if pie_survey_df.loc[i, "UserLanguage"] == "ZH-S":
		pie_survey_df.at[i, "UserLanguage"] = "ZH_S"
	if pie_survey_df.loc[i, "UserLanguage"] == "ZH-T":
		pie_survey_df.at[i, "UserLanguage"] = "ZH_T"
	if pie_survey_df.loc[i, "UserLanguage"] == "ES-ES":
		pie_survey_df.at[i, "UserLanguage"] = "ES_ES"

### Statistics

In [17]:
stats_groups_df: pd.DataFrame = pd.DataFrame()
for i in main_visitor_df["visitor_id"].values:
	sheet_name: str = group_sheet_name_base + " " + "{0:0=3d}".format(i)
	temp_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=sheet_name, index_col=0)
	col_name: str = "{0:0=3d}".format(i)
	stats_groups_df[col_name] = pd.Series(dtype=object)
	for j in temp_df.index.to_list():
		if "Frieze" in j:
			stats_groups_df.at[j, col_name] = (bool(temp_df.loc[j, "revisited"]), bool(temp_df.loc[j, "viewed_labels"]), bool(temp_df.loc[j, "took_photos"]))
		else:
			stats_groups_df.at[j, col_name] = (bool(temp_df.loc[j, "took_photos"]), False)
stats_groups_df = stats_groups_df.T

stats_artifacts_df: pd.DataFrame = pd.DataFrame()
for i in main_visitor_df["visitor_id"].values:
	sheet_name: str = indiv_sheet_name_base + " " + "{0:0=3d}".format(i)
	temp_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=sheet_name, index_col=0)
	col_name: str = "{0:0=3d}".format(i)
	stats_artifacts_df[col_name] = pd.Series(dtype=object)
	for j in temp_df.index.to_list():
		if "Frieze" in j:
			stats_artifacts_df.at[j, col_name] = (bool(temp_df.loc[j, "visited"]), bool(temp_df.loc[j, "viewed_labels"]), bool(temp_df.loc[j, "took_photos"]))
		else:
			stats_artifacts_df.at[j, col_name] = (bool(temp_df.loc[j, "revisited"]), bool(temp_df.loc[j, "viewed_labels"]), bool(temp_df.loc[j, "took_photos"]))
stats_artifacts_df = stats_artifacts_df.T

### Venn Diagrams

In [18]:
venn_visitor_df: pd.DataFrame = main_visitor_df.copy()

venn_lectern_counts: dict = venn_visitor_df["lecterns_visited"].value_counts()
venn_lectern_none_count: int = venn_lectern_counts["N"]
venn_lectern_counts = {
	"100": venn_lectern_counts["W"],
	"010": venn_lectern_counts["M"],
	"001": venn_lectern_counts["E"],
	"110": venn_lectern_counts["WM"],
	"101": venn_lectern_counts["WE"],
	"011": venn_lectern_counts["ME"],
	"111": venn_lectern_counts["WME"]
}

# first is lecterns, second is labels, third is photos
venn_engagement_counts = {
	"100": 0,
	"010": 0,
	"001": 0,
	"110": 0,
	"101": 0,
	"011": 0,
	"111": 0
}
venn_engagement_none_count = 0

for idx, row in venn_visitor_df.iterrows():
	visitor_id: str = "{0:0=3d}".format(row["visitor_id"])
	group_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=group_sheet_name_base + " " + visitor_id, index_col=0)
	indiv_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=indiv_sheet_name_base + " " + visitor_id, index_col=0)
	read_lectern: bool = row["lecterns_visited"] != "N"
	read_label: bool = False
	took_photo: bool = False
	for i in group_df.index.to_list():
		if group_df.loc[i, "viewed_labels"] == True:
			read_label = True
		if group_df.loc[i, "took_photos"] == True:
			took_photo = True
	for i in indiv_df.index.to_list():
		if indiv_df.loc[i, "viewed_labels"] == True:
			read_label = True
		if indiv_df.loc[i, "took_photos"] == True:
			took_photo = True
	if read_lectern:
		if read_label:
			if took_photo:
				venn_engagement_counts["111"] += 1
			else:
				venn_engagement_counts["110"] += 1
		elif took_photo:
			venn_engagement_counts["101"] += 1
		else:
			venn_engagement_counts["100"] += 1
	elif read_label:
		if took_photo:
			venn_engagement_counts["011"] += 1
		else:
			venn_engagement_counts["010"] += 1
	elif took_photo:
		venn_engagement_counts["001"] += 1
	else:
		venn_engagement_none_count += 1

## Constants

### Bar Charts

In [19]:
BAR_BEHAVIOR_ANALYSIS_COLS: list[str] = [
	"chatted_with_visitors",
	"chatted_with_staff",
	"sat_on_bench",
	"split_from_group",
	"used_phone",
	"used_museum_guide",
	"used_headphones"
]

BAR_SUBBEHAVIOR_ANALYSIS_COLS: list[str] = [
	"read_lectern",
	"read_label",
	"took_photos",
	"used_headphones"
]

BAR_SLIP_ROOM_ANALYSIS_COLS: list[str] = [
	"b18_viewed_labels",
	"b18_touched_casts",
	"b18_took_photos",
	"a18_viewed_labels",
	"a18_took_videos",
	"a18_took_photos"
]

BAR_BEHAVIOR_ENUM_CLASSES: list[str] = [
	BehaviorTitles
]

BAR_SLIP_ROOM_ENUM_CLASSES: list[str] = [
	SlipRoomTitles
]

bar_a_count: int = 0
bar_b_count: int = 0
for i in range(bar_visitor_df.shape[0]):
	if bar_visitor_df.isnull().loc[i, "18a_total_time"] == False:
		bar_a_count += 1
	if bar_visitor_df.isnull().loc[i, "18b_total_time"] == False:
		bar_b_count += 1

bar_max_count: int = (bar_a_count if bar_a_count > bar_b_count else bar_b_count)

### Box Plots

In [20]:
regex: re.Pattern = re.compile(r"^(.+?): - ([^:]+):(.+)$")
box_survey_question_data: dict[str, tuple[str, str, str]] = {}

for i in main_survey_question_names.keys():
	match = regex.match(main_survey_question_names[i])
	if match:
		question_name: str = match.group(1)[19:]
		box_survey_question_data[i] = (question_name[0].upper() + question_name[1:], match.group(2).title(), match.group(3).title())

BOX_OBSERVATION_TITLES: list = [
	"Time Spent in Room 18, 18a and 18b",
	"Time Spent in Room 18"
]

BOX_SLIP_ROOM_TITLES: list = [
	"Room 18a",
	"Room 18b"
]

BOX_GROUP_TITLES: list = [
	"South Metopes II-V",
	"West Pediments",
	"South Metopes VI-IX",
	"South Metopes XXVI-XXIX",
	"East Pediments",
	"South Metopes XXX-XXXII",
	"Frieze Section 1",
	"Frieze Section 2",
	"Frieze Section 3",
	"Frieze Section 4",
	"Frieze Section 5"
]

BOX_OBSERVATION_ANALYSIS_COLS: list[str] = [
	"total_time",
	"18_total_time"
]

BOX_SLIP_ROOM_ANALYSIS_COLS: list[str] = [
	"18a_total_time",
	"18b_total_time"
]

BOX_OBSERVATION_ENUM_CLASSES: list[Enum] = []

BOX_GROUP_ANALYSIS_COLS: list[str] = [
	"South Metopes II-V",
	"West Pediments",
	"South Metopes VI-IX",
	"South Metopes XXVI-XXIX",
	"East Pediments",
	"South Metopes XXX-XXXII",
	"Frieze Section 1",
	"Frieze Section 2",
	"Frieze Section 3",
	"Frieze Section 4",
	"Frieze Section 5"
]

BOX_SURVEY_ANALYSIS_COLS_21: list[str] = [
	"Q21_1",
	"Q21_2",
	"Q21_3",
	"Q21_4",
	"Q21_5",
]
BOX_SURVEY_ANALYSIS_COLS_22: list[str] = [
	"Q22_1",
]
BOX_SURVEY_ANALYSIS_COLS_23: list[str] = [
	"Q23_1"
]
BOX_SURVEY_ANALYSIS_COLS_KD: list[str] = [
	"knowledge_difference"
]

### Bubble Plot

In [21]:
# dictionary of areas to investigate and their coordinates on the map, incluuding the positioning of the label
BUBBLE_OBJECT_GROUP_NAMES: dict[str, tuple[int, int, bool]] = {
	"South Metopes II-V": (60, 400, False),
	"West Pediments": (50, 285, False),
	"South Metopes VI-IX": (60, 170, True),
	"South Metopes XXVI-XXIX": (525, 170, True),
	"East Pediments": (535, 285, False),
	"South Metopes XXX-XXXII": (525, 400, False),
	"Frieze Section 1": (190, 340, False),
	"Frieze Section 2": (160, 230, False),
	"Frieze Section 3": (290, 230, True),
	"Frieze Section 4": (420, 230, False),
	"Frieze Section 5": (400, 340, False)
}

# {name: (x, y, visitor %, avg dwell time, labeldirection)}
bubble_spots: dict[str, tuple[float, float, float, float, bool]] = {}
bubble_visit_totals: dict[str, int] = {}
bubble_max_visitors: float = 0
bubble_max_dwell_time: float = 0

# load data into dictionary
for group, coords in BUBBLE_OBJECT_GROUP_NAMES.items():
	total_visitors: int = 0
	total_dwell_time: int = 0
	for visitor_id, group_df in bubble_groups_dfs.items():
		row = group_df.loc[group]
		if row.empty:
			raise Exception("Error in row fetching: " + group + ", sheet: " + visitor_id)
		if row.isnull()["visit_order"] == False:
			total_visitors += 1
			total_dwell_time += row["dwell_time"]
	visit_percent: float = float(total_visitors) / num_visitors
	avg_dwell_time: float = float(total_dwell_time) / total_visitors
	if visit_percent > bubble_max_visitors:
		bubble_max_visitors = visit_percent
	if avg_dwell_time > bubble_max_dwell_time:
		bubble_max_dwell_time = avg_dwell_time

	bubble_spots[group] = (coords[0], coords[1], visit_percent, avg_dwell_time, coords[2])
	bubble_visit_totals[group] = total_visitors

### Pie Charts

In [22]:
# lists of columns to analyze and snums to transform labels
PIE_OBSERVATION_ANALYSIS_COLS: list = [
	"group_comp",
	"gender",
	"room_order",
	"lecterns_visited",
	"visitor_type",
	"first_turn_direction"
]
PIE_OBSERVATION_LABELLESS_ANALYSIS_COLS: list = [
	"room_order"
]

PIE_OBSERVATION_ENUM_CLASSES: list = [
	RoomOrder,
	LecternsViewed
]

PIE_OBSERVATION_TITLES: list = [
	"Types of Visitor Groups (100 Obserations)",
	"Perceived Visitor Gender (100 Obserations)",
	"Order of Rooms Visited (100 Obserations)",
	"Lecterns Visited (100 Obserations)",
	"Types of Visitors (100 Obserations)",
	"Direction of Visitors' First Turn (100 Obserations)"
]
PIE_OBSERVATION_LABELLESS_TITLES: list = [
	"Order of Rooms Visited (100 Obserations)"
]

PIE_SURVEY_ANALYSIS_COLS: list = [
	"UserLanguage",
	"Q3",
	"Q4",
	"Q5",
	"Q6",
	"Q7",
	"Q8",
	"Q10",
	"Q12",
	"Q13",
	"Q14",
	"Q16",
	"Q19"
]

PIE_SURVEY_ENUM_CLASSES: list = [
	Languages
]

### Statistics

In [23]:
STATS_GROUP_ANALYSIS_COLS: list = [
	"South Metopes II-V",
	"West Pediments",
	"South Metopes VI-IX",
	"South Metopes XXVI-XXIX",
	"East Pediments",
	"South Metopes XXX-XXXII",
	"Frieze Section 1",
	"Frieze Section 2",
	"Frieze Section 3",
	"Frieze Section 4", 
	"Frieze Section 5"
]

STATS_ARTIFACT_ANALYSIS_COLS: list = [
	"South Metope II",
	"South Metope III",
	"South Metope IV",
	"West Pediment A*",
	"West Pediment A (back)*",
	"West Pediment H",
	"West Pediment L*",
	"West Pediment M",
	"West Pediment N*",
	"West Pediment N (back)",
	"West Pediment O",
	"West Pediment Q",
	"South Metope V",
	"South Metope VI",
	"South Metope VII",
	"South Metope VIII",
	"South Metope IX",
	"East Pediment A-C*",
	"East Pediment D*",
	"East Pediment D (back)",
	"East Pediment E and F",
	"East Pediment E,F (back)",
	"East Pediment G*",
	"East Pediment G (back)",
	"East Pediment K",
	"East Pediment L and M*",
	"East Pediment K,L,M (back)",
	"East Pediment O",
	"South Metope XXVI",
	"South Metope XXVII*",
	"South Metope XXVIII",
	"South Metope XXIX",
	"South Metope XXX",
	"South Metope XXXI",
	"South Metope XXXII",
	"South Frieze XXXI 78-79*",
	"East Frieze III 7-11*",
	"East Frieze IV 24-25*",
	"East Frieze III 3-35*",
	"North Frieze V 13*",
	"North Frieze XXVII 73-74*",
	"North Frieze XLVII 132-136*",
	"West Frieze 2-3*",
]

### Venn Diagrams

In [24]:
VENN_LECTERN_TITLE: str = "Lecterns Visited (100 Observations)"

VENN_LECTERN_LABELS: list[str] = [
	"West Lectern",
	"Middle Lecterns",
	"East Lectern"
]

VENN_ENGAGEMENT_TITLE: str = "Room 18 Engagement (100 Observations)"

VENN_ENGAGEMENT_LABELS: list[str] = [
	"Read Lecterns",
	"Read Labels",
	"Took Photos"
]

## Functions

### Bar Charts

In [25]:
def bar_plot_data(df: pd.DataFrame, analysis_cols: list, enum_classes: list, sort_bars: bool, title: str, x_max: int, export_path: str, show_chart: bool) -> None:
	plot_df: pd.DataFrame = df.copy()
	plot_df = plot_df[analysis_cols]

	fig: plt.Figure
	ax: plt.Axes
	fig, ax = plt.subplots(figsize=(10, 6))

	master_map: dict[str, str] = {}
	for enum_cls in enum_classes:
		for member in enum_cls:
			master_map[member.name] = str(member.value)
	plot_df = plot_df.rename(columns=master_map)

	percentage_data: pd.Series = plot_df.mean() * 100
	if sort_bars:
		percentage_data = percentage_data.sort_values()

	bars: plt.BarContainer = ax.barh(
		percentage_data.index,
		percentage_data.values,
		color='lightgreen',
		edgecolor='black'
	)

	ax.set_title(title, fontsize=14)
	ax.set_xlabel('# of Visitors', fontsize=12)
	ax.set_xlim(0, x_max)
	ax.grid(axis='x', linestyle='--', alpha=0.7)

	for bar in bars:
		width: float = bar.get_width()
		ax.text(
			width + 1,
			bar.get_y() + bar.get_height() / 2,
			str(int(width)),
			va='center',
			fontsize=10
		)

	plt.tight_layout()
	if show_chart:
		plt.show()

	fig.savefig(export_path, format="png", dpi=150)
	plt.close(fig)

### Box Plots

In [26]:
def box_plot_data(df: pd.DataFrame, analysis_cols: list, enum_classes: list, names: dict, titles: list, label: str, export_path: str, metadata_export_path: str, show_plots: bool) -> None:
	if names is None and len(analysis_cols) != len(titles):
		raise Exception("Analysis Columns and Titles do not have the same length.")
	elif names is None and titles is None:
		raise Exception("Required: names dictionary or titles list.")

	doc: fitz.Document = fitz.open()

	given_names: bool = names is not None

	page_width: int = 595
	page_height: int = 842

	margin: int = 50
	midpoint: int = page_height / 2

	master_map: dict[str, str] = {}
	for enum_cls in enum_classes:
		for member in enum_cls:
			master_map[member.name] = str(member.value)

	plot_df: pd.DataFrame = df.copy()

	metadata_text: str = ""

	for i, col in enumerate(analysis_cols):
		if i % 2 == 0:
			page: fitz.Page = doc.new_page(width=page_width, height=page_height)
		
		fig: plt.Figure
		ax: plt.Axes
		fig, ax = plt.subplots(figsize=(8, 5))

		df[col] = pd.to_numeric(df[col], errors='coerce')
		cleaned_data = df[(df[col] != 0) & (df[col].notna())][col]
		plt.boxplot(cleaned_data, vert=False)

		if given_names:
			plt.title(names[col][0] + "\n" + names[col][1] + " --- " + names[col][2])
		else:
			plt.title(titles[i])
		
		summary = cleaned_data.describe(percentiles=[.25, .5, .75])
		q1 = round(summary['25%'], 3)
		q3 = round(summary['75%'], 3)
		iqr = q3 - q1
		metadata_text += col + ":\n"
		metadata_text += "Min: " + str(round(summary['min'], 3)) + ", Q1: " + str(q1) + ", Median: " + str(round(summary['50%'], 3)) + ", Q3: " + str(q3) + ", Max: " + str(round(summary['max'], 3)) + ", IQR: " + str(iqr) + ", Mean: " + str(round(summary['mean'], 3)) + ", STD: " + str(round(summary['std'], 3)) + "\n\n"

		plt.xticks(rotation=45)
		plt.xlabel(label)
		plt.yticks([])
		plt.subplots_adjust(bottom=0.2, top=0.9)
		image_data: io.BytesIO = io.BytesIO()
		fig.savefig(image_data, format="png", dpi=150)
		if show_plots:
			plt.show(fig)
		image_data.seek(0)

		image_rect: fitz.Rect
		text_y: int
		if i % 2 == 0:
			image_rect = fitz.Rect(margin, margin, page_width - margin, midpoint - 40)
			text_y = midpoint - 20
		else:
			image_rect = fitz.Rect(margin, midpoint + 20, page_width - margin, page_height - margin - 40)
			text_y = page_height - 30
		
		page.insert_image(image_rect, stream=image_data.read())

		page.insert_text((margin, text_y), "Analysis for column '" + col + "'", fontsize=12)
		plt.close(fig)
	doc.save(export_path)
	doc.close()

	metadata_doc: fitz.Document = fitz.open()
	metadata_page: fitz.Page = metadata_doc.new_page()
	metadata_page.insert_text(
		fitz.Point(50, 50),
		metadata_text,
		fontsize=12
	)
	metadata_doc.save(metadata_export_path)
	metadata_doc.close()

In [27]:
def box_plot_group_data(df: pd.DataFrame, analysis_cols: list, names: dict, titles: list, label: str, plot_title: str, export_path: str, metadata_export_path: str, show_plots: bool, create_new_doc: bool) -> None:
	doc: fitz.Document = (fitz.open() if create_new_doc else fitz.open(export_path))
	page: fitz.Page = doc.new_page()

	given_names: bool = names is not None

	page_width: int = 595
	page_height: int = 842

	margin: int = 50
	midpoint: int = page_height / 2
	
	plot_df: pd.DataFrame = df.copy()

	fig: plt.Figure
	ax: plt.Axes
	fig, ax = plt.subplots(figsize=(8, 5))
	metadata_text: str = ""

	plot_data: list = []
	plot_labels: list = []

	for i, col in enumerate(analysis_cols):

		plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
		cleaned_data = plot_df[(plot_df[col] != 0) & (plot_df[col].notna())][col]
		plot_data.append(cleaned_data)

		if given_names:
			plot_labels.append(names[col][1] + " to " + names[col][2])
		else:
			plot_labels.append(titles[i])

		summary = cleaned_data.describe(percentiles=[.25, .5, .75])
		q1 = round(summary['25%'], 3)
		q3 = round(summary['75%'], 3)
		iqr = q3 - q1
		metadata_text += col + ":\n"
		metadata_text += "Min: " + str(round(summary['min'], 3)) + ", Q1: " + str(q1) + ", Median: " + str(round(summary['50%'], 3)) + ", Q3: " + str(q3) + ", Max: " + str(round(summary['max'], 3)) + ", IQR: " + str(iqr) + ", Mean: " + str(round(summary['mean'], 3)) + ", STD: " + str(round(summary['std'], 3)) + "\n\n"

	ax.boxplot(plot_data, vert=False, tick_labels=plot_labels)
	
	plt.xticks(rotation=45)
	plt.xlabel(label)
	if plot_title is None:
		ax.set_title(names[col][0])
	else:
		ax.set_title(plot_title)

	plt.tight_layout()
	# plt.subplots_adjust(bottom=0.2, top=0.9)
	image_data: io.BytesIO = io.BytesIO()
	fig.savefig(image_data, format="png", dpi=150)
	if show_plots:
		plt.show(fig)
	image_data.seek(0)

	image_rect: fitz.Rect
	text_y: int
	image_rect = fitz.Rect(margin, margin, page_width - margin, midpoint - 40)
	text_y = midpoint - 20
	page.insert_image(image_rect, stream=image_data.read())

	plt.close(fig)
	if create_new_doc:
		doc.save(export_path)
	else:
		doc.saveIncr()
	doc.close()

	metadata_doc: fitz.Document = (fitz.open() if create_new_doc else fitz.open(metadata_export_path))
	metadata_page: fitz.Page = metadata_doc.new_page()
	metadata_page.insert_text(
		fitz.Point(50, 50),
		metadata_text,
		fontsize=12
	)
	if create_new_doc:
		metadata_doc.save(metadata_export_path)
	else:
		metadata_doc.saveIncr()
	metadata_doc.close()

In [28]:
def box_plot_kd_data(df: pd.DataFrame, analysis_cols: list, names: dict, titles: list, label: str, plot_title: str, metadata_export_path: str, show_plots: bool, create_new_doc: bool) -> plt.Figure:
	given_names: bool = names is not None

	page_width: int = 595
	page_height: int = 842

	margin: int = 50
	midpoint: int = page_height / 2
	
	plot_df: pd.DataFrame = df.copy()

	fig: plt.Figure
	ax: plt.Axes
	fig, ax = plt.subplots(figsize=(8, 5))
	metadata_text: str = ""

	plot_data: list = []
	plot_labels: list = []

	for i, col in enumerate(analysis_cols):

		df[col] = pd.to_numeric(df[col], errors='coerce')
		cleaned_data = df[(df[col] != 0) & (df[col].notna())][col]
		plot_data.append(cleaned_data)

		if given_names:
			plot_labels.append(names[col][1] + " to " + names[col][2])
		else:
			plot_labels.append(titles[i])

		summary = cleaned_data.describe(percentiles=[.25, .5, .75])
		q1 = round(summary['25%'], 3)
		q3 = round(summary['75%'], 3)
		iqr = q3 - q1
		metadata_text += col + ":\n"
		metadata_text += "Min: " + str(round(summary['min'], 3)) + ", Q1: " + str(q1) + ", Median: " + str(round(summary['50%'], 3)) + ", Q3: " + str(q3) + ", Max: " + str(round(summary['max'], 3)) + ", IQR: " + str(iqr) + ", Mean: " + str(round(summary['mean'], 3)) + ", STD: " + str(round(summary['std'], 3)) + "\n\n"

	ax.boxplot(plot_data, vert=False, tick_labels=plot_labels)
	
	plt.xticks(rotation=45)
	plt.xlabel(label)
	if plot_title is None:
		ax.set_title(names[col][0])
	else:
		ax.set_title(plot_title)

	plt.tight_layout()

	if show_plots:
		plt.show(fig)

	metadata_doc: fitz.Document = (fitz.open() if create_new_doc else fitz.open(metadata_export_path))
	metadata_page: fitz.Page = metadata_doc.new_page()
	metadata_page.insert_text(
		fitz.Point(50, 50),
		metadata_text,
		fontsize=12
	)
	if create_new_doc:
		metadata_doc.save(metadata_export_path)
	else:
		metadata_doc.saveIncr()
	metadata_doc.close()

	return fig

### Bubble Plot

In [29]:
def bubble_create_overlay(data: dict, export_path: str, metadata_export_path: str, show_plot: bool) -> None:
	doc: fitz.Document = fitz.open(master_tracker_path)
	page: fitz.Page = doc[0]

	pixmap: fitz.Pixmap = page.get_pixmap(dpi=72)
	width: float = page.rect.width
	height: float = page.rect.height
   
	img_data: np.ndarray = np.frombuffer(pixmap.samples, dtype=np.uint8).reshape(
		pixmap.h, pixmap.w, pixmap.n
	)

	fig: plt.Figure
	ax_map: plt.Axes
	ax_leg: plt.Axes
	fig, (ax_map, ax_leg) = plt.subplots(
		1, 2, 
		figsize=(width / 72 + 2, height / 72),
		gridspec_kw={'width_ratios': [width/36, 2]},
		constrained_layout=True
	)

	ax_map.imshow(img_data, extent=[0, width, height, 0])

	metadata_text: str = ""

	norm: mcolors.Normalize = mcolors.Normalize(vmin=0, vmax=bubble_max_dwell_time)
	cmap: mcolors.Colormap = plt.get_cmap('YlOrRd')

	for name, (x, y, visitor_percent, avg_dwell_time, direction) in data.items():
		color: tuple[float, float, float, float] = cmap(norm(avg_dwell_time))
		size: float = ((visitor_percent / bubble_max_visitors) * 35 + 10)
		metadata_text += name + ":\nvisitor %: " + str(int(visitor_percent * 100)) + "%, average dwell: " + str(float(int(avg_dwell_time * 100)) / 100.0) + " sec\n\n"

		ax_map.scatter(x, y - 75, s=size**2, c=[color], alpha=0.8, edgecolors="black", zorder=5)
		ax_map.text(
			x,
			y - ((size + 15) * (1 if direction else -1)) - 75,
			name,
			fontsize=9,
			ha="center",
			fontweight="bold",
			zorder=6,
			bbox=dict(
				facecolor='white',
				alpha=0.8,
				edgecolor='none',
				boxstyle='round,pad=0.3'
			)
		)

	sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
	sm.set_array([])
	cbar = fig.colorbar(
		sm,
		ax=ax_map,
		shrink=0.5,
		aspect=20,
		pad=0.08,
		location="left"
	)
	cbar.set_label('Avg Dwell Time (sec)', rotation=90, labelpad=15)
	
	ax_map.set_xlim(0, width)
	ax_map.set_ylim(height, 0)
	ax_map.axis("off")

	ax_leg.axis("off")
	ax_leg.set_xlim(0, 10)
	ax_leg.set_ylim(0, 100)

	legend_values = [100, 75, 50, 25]
	
	current_y = 70

	max_size_radius: float = 0
	
	for val in legend_values:
		size_radius: float = ((val / 100) / bubble_max_visitors * 35 + 10)
		if max_size_radius == 0 or size_radius > max_size_radius:
			max_size_radius = size_radius
		
		ax_leg.scatter(5, current_y, s=size_radius**2, c="gray", alpha=0.8, edgecolors="black", zorder=5)
		ax_leg.text(5.5+(size_radius / 9), current_y, str(val), va="center", fontsize=10)

		current_y -= (size_radius / 7 + 10)
	
	ax_leg.text(max_size_radius / 4 + 4, 50, '# of Visitors Observed that Visited Artifact', va="center", rotation=270)

	fig.suptitle("Room 18 Visitor Bubble Plot", y=1)
	plt.savefig(export_path, dpi=300, bbox_inches='tight')
	if show_plot:
		plt.show()
	
	plt.close(fig)
	doc.close()

	metadata_doc: fitz.Document = fitz.open()
	metadata_page: fitz.Page = metadata_doc.new_page()
	metadata_page.insert_text(
		fitz.Point(50, 50),
		metadata_text,
		fontsize=12
	)
	metadata_doc.save(metadata_export_path)
	metadata_doc.close()

### Pie Charts

In [30]:
def pie_plot_data(df: pd.DataFrame, analysis_cols: list, enum_classes: list, names: pd.Series, titles: str, show_numbers: bool, export_path: str, show_charts: bool, create_new_doc: bool) -> None:
	if names is None and len(analysis_cols) != len(titles):
		raise Exception("Analysis Columns and Titles do not have the same length.")
	elif names is None and titles is None:
		raise Exception("Required: names dictionary or titles list.")

	doc: fitz.Document = (fitz.open() if create_new_doc else fitz.open(export_path))

	given_names: bool = names is not None

	page_width: int = 595
	page_height: int = 842

	margin: int = 50
	midpoint: int = page_height / 2

	master_map: dict[str, str] = {}
	for enum_cls in enum_classes:
		for member in enum_cls:
			master_map[member.name] = str(member.value)

	plot_df: pd.DataFrame = df.copy()

	for i, col in enumerate(analysis_cols):
		if i % 2 == 0:
			page: fitz.Page = doc.new_page(width=page_width, height=page_height)
		
		plot_df[col] = plot_df[col].map(lambda x: master_map.get(x, x))
		data_counts: pd.Series = plot_df[col].value_counts()
		
		if show_numbers:
			fig: go.Figure = go.Figure(data=[go.Pie(
				labels=list(data_counts.index), 
				values=list(data_counts.values),
				textinfo='label+value',
				insidetextorientation='radial',
			)])
		else:
			fig: go.Figure = go.Figure(data=[go.Pie(
				labels=list(data_counts.index), 
				values=list(data_counts.values),
				textinfo='label',
				insidetextorientation='radial',
			)])
		
		if given_names:
			fig.update_layout(
				title={
					'text': "'" + names[col] + "'",
					'y':0.95,
					'x':0.5,
					'xanchor': 'center',
					'yanchor': 'top'
				},
				width=800,
				margin=dict(t=80, b=10, l=10, r=10)
			)
		else:
			fig.update_layout(
				title={
					'text': titles[i],
					'y':0.95,
					'x':0.5,
					'xanchor': 'center',
					'yanchor': 'top'
				},
				width=800,
				margin=dict(t=50, b=10, l=10, r=10)
			)

		image_bytes: bytes = pio.to_image(fig, format="png", width=800, height=500, scale=2)
		
		if show_charts:
			fig.show()

		image_rect: fitz.Rect
		if i % 2 == 0:
			image_rect = fitz.Rect(margin, margin, page_width - margin, midpoint - 40)
		else:
			image_rect = fitz.Rect(margin, midpoint + 20, page_width - margin, page_height - margin - 40)
		
		page.insert_image(image_rect, stream=image_bytes)
		

	if create_new_doc:
		doc.save(export_path)
	else:
		doc.saveIncr()
	doc.close()

In [31]:
def categorize_value(val: str) -> str:
	has_a = 'A' in val
	has_b = 'B' in val
	
	if has_a and has_b:
		return 'Both A and B'
	elif has_a:
		return 'Only A'
	elif has_b:
		return 'Only B'
	else:
		return 'Neither'

def pie_plot_room_data(df: pd.DataFrame, export_path: str, show_chart: bool) -> None:
	doc: fitz.Document = fitz.open(export_path)
	
	page_width: int = 595
	page_height: int = 842
	
	margin: int = 50
	midpoint: int = page_height / 2

	page: fitz.Page = doc.new_page(width=page_width, height=page_height)

	room_data: pd.Series = df['room_order'].apply(categorize_value)
	data_counts: pd.DataFrame =  room_data.value_counts()
	
	fig: go.Figure = go.Figure(data=[go.Pie(
		labels=list(data_counts.index), 
		values=list(data_counts.values),
		textinfo='label+value',
		insidetextorientation='radial',
	)])
	fig.update_layout(
		title={
			'text': "Visitation of Slip Rooms (100 Observations)",
			'y':0.95,
			'x':0.5,
			'xanchor': 'center',
			'yanchor': 'top'
		},
		width=800,
		margin=dict(t=50, b=10, l=10, r=10)
	)

	image_bytes: bytes = pio.to_image(fig, format="png", width=800, height=500, scale=2)

	if show_chart:
		fig.show()

	image_rect: fitz.Rect = fitz.Rect(margin, margin, page_width - margin, midpoint - 40)
	page.insert_image(image_rect, stream=image_bytes)

	doc.saveIncr()
	doc.close()

### Statistics

In [32]:
def stats_collect_data(df: pd.DataFrame, analysis_cols: list[str], num_visitors: int) -> dict[str, tuple[float, float]]:
	engagement: dict[str, tuple[float, float]] = {}
	for col in analysis_cols:
		engagement_count: int = 0
		visitor_count: int = 0
		for cell in df[col].values:
			cell_counted: bool = False
			for element in cell:
				if element == True:
					if not cell_counted:
						visitor_count += 1
						cell_counted = True
					engagement_count += 1
		engagement[col] = (engagement_count, visitor_count)
	return engagement

def stats_plot_data(data: dict[str, tuple[float, float]], start_string: str, export_path: str, show_data: bool) -> None:
	doc: fitz.Document = fitz.open()
	page: fitz.Page = doc.new_page()

	margin: int = 50
	
	sorted_keys: list[str] = sorted(
		data,
		key=lambda k: (-sum(data[k]), k)
	)

	current_rank: int = 0
	next_rank: int = 1
	previous_score: int = None
	text_content: str = "\n\n"
	for i, key in enumerate(sorted_keys):
		current_score: int = sum(data[key])
		if current_score != previous_score:
			previous_score = current_score
			current_rank = next_rank
			next_rank += 1
		else:
			next_rank += 1
		text_content += str(current_rank) + ") " + str(key) + ": " + str(data[key][0]) + " (engagements) + " + str(data[key][1]) + " (visitors engaged) \n"
	
	page.insert_text(
		fitz.Point(margin, margin),
		start_string + ":",
		fontsize=16
	)
	page.insert_text(
		fitz.Point(margin, margin),
		text_content,
		fontsize=12
	)

	if show_data:
		print(text_content)

	doc.save(export_path)
	doc.close()

### Venn Diagrams

In [33]:
def venn_plot_data(counts: dict, none_count: int, title: str, labels: list, export_path: str, create_new_doc: bool) -> None:
	doc: fitz.Document = (fitz.open() if create_new_doc else fitz.open(export_path))
	
	page_width: int = 595
	page_height: int = 842
	
	margin: int = 50
	midpoint: int = page_height / 2

	page: fitz.Page = doc.new_page(width=page_width, height=page_height)
	
	fig: plt.Figure = plt.figure(figsize=(10, 6))
	venn = venn3(subsets=counts, set_labels=labels, layout_algorithm=DefaultLayoutAlgorithm(fixed_subset_sizes=(1,1,1,1,1,1,1)))
	
	for text in venn.set_labels:
		if text: text.set_fontsize(14)
	for text in venn.subset_labels:
		if text: text.set_fontsize(12)
	
	plt.title(title, fontsize=16)
	annotation: plt.Text = plt.annotate(
		"None: " + str(none_count),
		xy=(0.1, 0.05),
		xycoords='figure fraction',
		fontsize=12,
		fontweight='bold',
		bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=1)
	)
	
	img_data: io.BytesIO = io.BytesIO()
	plt.savefig(img_data, format="png", bbox_inches='tight')
	img_data.seek(0)
	image_bytes: bytes = img_data.read()

	image_rect: fitz.Rect = fitz.Rect(margin, margin, page_width - margin, midpoint - 40)
	page.insert_image(image_rect, stream=image_bytes)

	if create_new_doc:
		doc.save(export_path)
	else:
		doc.saveIncr()
	doc.close()
	plt.close(fig)

## Execution

### Bar Charts

In [34]:
bar_plot_data(bar_visitor_df, BAR_BEHAVIOR_ANALYSIS_COLS, BAR_BEHAVIOR_ENUM_CLASSES, True, 'Visitors that Displayed Notable Behaviors', num_visitors, bar_behavior_export_path, False)

In [35]:
bar_plot_data(bar_visitor_df, BAR_SLIP_ROOM_ANALYSIS_COLS, BAR_SLIP_ROOM_ENUM_CLASSES, False, 'Visitors that Displayed Behaviors in Rooms 18a and 18b', bar_max_count, bar_slip_room_export_path, False)

In [36]:
bar_plot_data(bar_visitor_df, BAR_SUBBEHAVIOR_ANALYSIS_COLS, BAR_BEHAVIOR_ENUM_CLASSES, False, 'Visitors that Displayed Behaviors in Room 18', num_visitors, bar_subbehavior_export_path, False)

### Box Plots

In [37]:
box_plot_data(box_visitor_df, BOX_OBSERVATION_ANALYSIS_COLS, BOX_OBSERVATION_ENUM_CLASSES, None, BOX_OBSERVATION_TITLES, "Time (min)", box_observation_export_path, box_observation_metadata_export_path, False)

In [38]:
box_plot_group_data(box_groups_df, BOX_GROUP_ANALYSIS_COLS, None, BOX_GROUP_TITLES, "Time (sec)", "Time Spent at Artifact Sections", box_group_export_path, box_group_metadata_export_path, False, True)

In [39]:
box_plot_group_data(box_visitor_df, BOX_SLIP_ROOM_ANALYSIS_COLS, None, BOX_SLIP_ROOM_TITLES, "Time (min)", "Visitor Time Spent in Slip Rooms", box_observation_export_path, box_slip_room_metadata_export_path, False, False)

In [40]:
box_plot_group_data(box_survey_df, BOX_SURVEY_ANALYSIS_COLS_21, box_survey_question_data, None, "Ratings", None, box_survey_export_path, box_survey_metadata_export_path, False, True)

In [41]:
plt_A: plt.Figure = box_plot_kd_data(box_survey_df, BOX_SURVEY_ANALYSIS_COLS_22, None, [""], "Ratings", r"$\boldsymbol{A.}$ Rate your knowledge on the Parthenon before visiting", box_survey_kd_metadata_export_path, False, True)
plt_B: plt.Figure = box_plot_kd_data(box_survey_df, BOX_SURVEY_ANALYSIS_COLS_23, None, [""], "Ratings", r"$\boldsymbol{B.}$ Rate your knowledge on the Parthenon after visiting", box_survey_kd_metadata_export_path, False, False)
plt_C: plt.Figure = box_plot_kd_data(box_survey_df, BOX_SURVEY_ANALYSIS_COLS_KD, None, [""], "Difference of Ratings", r"$\boldsymbol{C.}$ Knowledge Difference, Before and After", box_survey_kd_metadata_export_path, False, False)

doc = fitz.open()
page = doc.new_page(pno=-1, width=595, height=842) # A4 dimensions

# 2. Convert Matplotlib figures to image bytes
images = []
for fig in [plt_A, plt_B, plt_C]:
	img_data = io.BytesIO()
	fig.savefig(img_data, format='png', bbox_inches='tight', dpi=150)
	images.append(img_data.getvalue())

# 3. Define the Rectangles (Layout)
# fitz.Rect(x0, y0, x1, y1)

# Row 1: Two plots side-by-side
rect1 = fitz.Rect(50, 50, 290, 290)   # Top Left
rect2 = fitz.Rect(300, 50, 540, 290)  # Top Right

# Row 2: One plot centered below
rect3 = fitz.Rect(140, 235, 450, 485) # Bottom Center

# 4. Insert the images into the rectangles
page.insert_image(rect1, stream=images[0])
page.insert_image(rect2, stream=images[1])
page.insert_image(rect3, stream=images[2])

# 5. Save
doc.save(box_survey_kd_export_path)
doc.close()
plt.close(plt_A)
plt.close(plt_B)
plt.close(plt_C)

In [42]:
doc: fitz.Document = fitz.open(box_survey_kd_export_path)
page: fitz.Page = doc.new_page()

page_width: int = 595
page_height: int = 842

margin: int = 50
midpoint: int = page_height / 2

plot_df: pd.DataFrame = box_survey_df.copy()

fig: plt.Figure
ax: plt.Axes
fig, ax = plt.subplots(figsize=(8, 5))
metadata_text: str = ""

plot_data: list = []
plot_labels: list = []

for i, col in {"After Visiting": "Q23_1", "Before Visiting": "Q22_1"}.items():

	plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
	cleaned_data = plot_df[(plot_df[col] != 0) & (plot_df[col].notna())][col]
	plot_data.append(cleaned_data)

	plot_labels.append(i)

ax.boxplot(plot_data, vert=False, tick_labels=plot_labels)

plt.xticks(rotation=45)
plt.xlabel("Ratings")
ax.set_title("Rate your knowledge on the Parthenon")

plt.tight_layout()
# plt.subplots_adjust(bottom=0.2, top=0.9)
image_data: io.BytesIO = io.BytesIO()
fig.savefig(image_data, format="png", dpi=150)
# plt.show(fig)
image_data.seek(0)

image_rect: fitz.Rect
text_y: int
image_rect = fitz.Rect(margin, margin, page_width - margin, midpoint - 40)
text_y = midpoint - 20
page.insert_image(image_rect, stream=image_data.read())

plt.close(fig)
doc.saveIncr()
doc.close()

### Bubble Plot

In [43]:
bubble_create_overlay(bubble_spots, bubble_export_path, bubble_metadata_export_path, False)

In [44]:
total: int = 0
groups_counted: int = 0
for group, visits in bubble_visit_totals.items():
	if "Frieze" in group:
		total += visits
		groups_counted += 1
print("Average Number of Friezes Viewed by Visitors: " + str(total / num_visitors))

Average Number of Friezes Viewed by Visitors: 1.5


### Pie Charts

In [45]:
pie_plot_data(pie_visitor_df, PIE_OBSERVATION_ANALYSIS_COLS, PIE_OBSERVATION_ENUM_CLASSES, None, PIE_OBSERVATION_TITLES, True, pie_observation_export_path, False, True)
pie_plot_data(pie_visitor_df, PIE_OBSERVATION_LABELLESS_ANALYSIS_COLS, PIE_OBSERVATION_ENUM_CLASSES, None, PIE_OBSERVATION_LABELLESS_TITLES, False, pie_observation_export_path, False, False)

In [ ]:
pie_plot_data(pie_survey_df, PIE_SURVEY_ANALYSIS_COLS, PIE_SURVEY_ENUM_CLASSES, pie_survey_question_names, None, True, pie_survey_export_path, False, True)

In [ ]:
pie_plot_room_data(pie_visitor_df, pie_observation_export_path, False)

### Statistics

In [ ]:
stats_group_data: dict[str, tuple[float, float]] = stats_collect_data(stats_groups_df, STATS_GROUP_ANALYSIS_COLS, num_visitors)
stats_plot_data(stats_group_data, "Most Engaging Sections", stats_group_export_path, False)

In [ ]:
stats_artifacts_data: dict[str, tuple[float, float]] = stats_collect_data(stats_artifacts_df, STATS_ARTIFACT_ANALYSIS_COLS, num_visitors)
stats_plot_data(stats_artifacts_data, "Most Engaging Artifacts", stats_artifact_export_path, False)

### Venn Diagrams

In [ ]:
lectern_counts: dict = venn_visitor_df["lecterns_visited"].value_counts()
none_count: int = lectern_counts["N"]
lectern_counts = {
	"100": lectern_counts["W"],
	"010": lectern_counts["M"],
	"001": lectern_counts["E"],
	"110": lectern_counts["WM"],
	"101": lectern_counts["WE"],
	"011": lectern_counts["ME"],
	"111": lectern_counts["WME"]
}
venn_plot_data(lectern_counts, none_count, "Lecterns Visited (100 Observations)", ["West Lectern", "Middle Lecterns", "East Lectern"], venn_observation_export_path, True)

In [ ]:
# first is lecterns, second is labels, third is photos
engagement_counts = {
	"100": 0,
	"010": 0,
	"001": 0,
	"110": 0,
	"101": 0,
	"011": 0,
	"111": 0
}
none_count = 0

for idx, row in venn_visitor_df.iterrows():
	visitor_id: str = "{0:0=3d}".format(row["visitor_id"])
	group_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=group_sheet_name_base + " " + visitor_id, index_col=0)
	indiv_df: pd.DataFrame = pd.read_excel(visitor_xlsx, sheet_name=indiv_sheet_name_base + " " + visitor_id, index_col=0)
	read_lectern: bool = row["lecterns_visited"] != "N"
	read_label: bool = False
	took_photo: bool = False
	for i in group_df.index.to_list():
		if group_df.loc[i, "viewed_labels"] == True:
			read_label = True
		if group_df.loc[i, "took_photos"] == True:
			took_photo = True
	for i in indiv_df.index.to_list():
		if indiv_df.loc[i, "viewed_labels"] == True:
			read_label = True
		if indiv_df.loc[i, "took_photos"] == True:
			took_photo = True
	if read_lectern:
		if read_label:
			if took_photo:
				engagement_counts["111"] += 1
			else:
				engagement_counts["110"] += 1
		elif took_photo:
			engagement_counts["101"] += 1
		else:
			engagement_counts["100"] += 1
	elif read_label:
		if took_photo:
			engagement_counts["011"] += 1
		else:
			engagement_counts["010"] += 1
	elif took_photo:
		engagement_counts["001"] += 1
	else:
		none_count += 1

venn_plot_data(engagement_counts, none_count, "Room 18 Engagement (100 Observations)", ["Read Lecterns", "Read Labels", "Took Photos"], venn_observation_export_path, False)